Librerias

In [4]:
!pip install -q kaggle
!pip install -q pillow
!pip install -q pandas
!pip install -q matplotlib
!pip install -q numpy
!pip install imagehash --quiet

In [5]:
import pandas as pd
import os
import imagehash
from PIL import Image, UnidentifiedImageError
from procesamiento import dividir_train_val_test, calcular_hash, marcar_duplicado_verificado, son_realmente_duplicados



Cosas necesarias para tener el  train test

In [6]:
N_POR_CLASE = 600
SEMILLA = 42

RUTA_DATASET = r"archive/asl_alphabet_train/asl_alphabet_train"

datos_img = []

for clase in sorted(os.listdir(RUTA_DATASET)):

    carpeta = os.path.join(RUTA_DATASET, clase)

    if not os.path.isdir(carpeta):
        continue

    for archivo in os.listdir(carpeta):

        ruta = os.path.join(carpeta, archivo)

        try:

            imagen = Image.open(ruta)

            ancho, alto = imagen.size

            datos_img.append({
                "clase": clase,
                "archivo": archivo,
                "ruta": ruta,
                "ancho": ancho,
                "alto": alto,
                "modo": imagen.mode,
                "formato": imagen.format
            })

        except Exception as e:

            print(f"No se pudo abrir {ruta}")

datos_img = pd.DataFrame(datos_img)

print("Cantidad total de imágenes:")
print(len(datos_img))

datos_img.head()

submuestra = (
    datos_img.groupby("clase", group_keys=False)
    .apply(lambda x: x.sample(n=min(N_POR_CLASE, len(x)), random_state=SEMILLA))
    .reset_index(drop=True)
)

def calcular_hash(ruta, hash_size=16):
    try:
        img = Image.open(ruta)
        # hash_size mayor = huella más detallada = menos colisiones falsas
        return str(imagehash.phash(img, hash_size=hash_size))
    except Exception:
        return None
    
submuestra["phash"] = submuestra["ruta"].apply(lambda r: calcular_hash(r, hash_size=16))

indices_duplicados_reales = []
for phash_val, grupo in submuestra[submuestra.duplicated(subset="phash", keep=False)].groupby("phash"):
    indices_duplicados_reales.extend(marcar_duplicado_verificado(grupo))

print(f"\nDuplicados confirmados con verificación de píxeles: {len(indices_duplicados_reales)}")

submuestra_limpia = submuestra.drop(index=indices_duplicados_reales).reset_index(drop=True)



# Uso directo
train_df, val_df, test_df = dividir_train_val_test(
    df=submuestra_limpia, 
    col_clase="clase", 
    semilla=SEMILLA
)

print(train_df)
print(val_df)
print(test_df)

Cantidad total de imágenes:
87000


C:\Users\Usuario Preinstalado\AppData\Local\Temp\ipykernel_2636\3627686906.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(N_POR_CLASE, len(x)), random_state=SEMILLA))



Duplicados confirmados con verificación de píxeles: 9
Train: 12173 (70.0%) | Val: 2609 (15.0%) | Test: 2609 (15.0%)
       clase        archivo  \
1702       C       C829.jpg   
12643      V      V1980.jpg   
13215      W       W182.jpg   
6194       K        K57.jpg   
17213  space  space2175.jpg   
...      ...            ...   
14475      Y      Y1713.jpg   
12428      U      U2105.jpg   
12593      V       V262.jpg   
17170  space   space244.jpg   
10612      R      R2419.jpg   

                                                    ruta  ancho  alto modo  \
1702   archive/asl_alphabet_train/asl_alphabet_train\...    200   200  RGB   
12643  archive/asl_alphabet_train/asl_alphabet_train\...    200   200  RGB   
13215  archive/asl_alphabet_train/asl_alphabet_train\...    200   200  RGB   
6194   archive/asl_alphabet_train/asl_alphabet_train\...    200   200  RGB   
17213  archive/asl_alphabet_train/asl_alphabet_train\...    200   200  RGB   
...                                       